# **Carga de los datos**

In [1]:
import os

base_dir = './cats_and_dogs_small'

train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')

# Directorio con las imágenes de training
train_cats_dir = os.path.join(train_dir, 'cats')
train_dogs_dir = os.path.join(train_dir, 'dogs')

# Directorio con las imágenes de validación
validation_cats_dir = os.path.join(validation_dir, 'cats')
validation_dogs_dir = os.path.join(validation_dir, 'dogs')

# Directorio con las imágenes de test
test_cats_dir = os.path.join(test_dir, 'cats')
test_dogs_dir = os.path.join(test_dir, 'dogs')

# **Data Augmentation**

In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(train_dir,
                                                    batch_size=20,
                                                    class_mode='binary',
                                                    target_size=(150, 150))
validation_generator = validation_datagen.flow_from_directory(validation_dir,
                                                                batch_size=20,
                                                                class_mode='binary',
                                                                target_size=(150, 150))
test_generator = test_datagen.flow_from_directory(test_dir,
                                                    batch_size=20,
                                                    class_mode='binary',
                                                    target_size=(150, 150))

Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


# **Obtención de una red ya entrenada**

Las redes se obtienen con el módulo *tensorflow.keras.applications*. Para este ejemplo de utilizará la red **VGG16** adaptandola al tamaño de nuestras imágenes de entrada.

In [5]:
from tensorflow.keras.applications import VGG16

pre_trained_model = VGG16(input_shape=(150, 150, 3), 
                                include_top=False, 
                                weights='imagenet')

Para esta práctica, aplicaremos el concepto de *fine-tuning*, para ello entrenaremos sólo la última capa convolucional que es *block_5* de la siguiente manera.

In [6]:
pre_trained_model.trainable = True

set_trainable = False

for layer in pre_trained_model.layers:
    if layer.name == 'block5_conv1':
        set_trainable = True
    if set_trainable:
        layer.trainable = True
    else:
        layer.trainable = False

In [7]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Dense

modelIFE = Sequential()

modelIFE.add(pre_trained_model)
modelIFE.add(Flatten())
modelIFE.add(Dense(256, activation='relu'))
modelIFE.add(Dense(1, activation='sigmoid'))

modelIFE.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,812,353 (64.13 MB)

 Trainable params: 9,177,089 (35.01 MB)

 Non-trainable params: 7,635,264 (29.13 MB)

En Keras los modelos se considerán cómo capas, así que lo que tendríamos que hacer es crear un modelo aparte que es el que va a ser las capas densas cómo clasificador y luego entrenaríamos el modelo.

# **Entrenamiento de nuestra red resultante**

In [8]:
from tensorflow.keras.optimizers import RMSprop

modelIFE.compile(optimizer=RMSprop(learning_rate=1e-4),
              loss='binary_crossentropy',
              metrics=['acc'])

In [9]:
modelIFE.fit(train_generator, epochs=10, 
          validation_data=validation_generator, 
          steps_per_epoch=100, 
          validation_steps=50)

Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 105s 1s/step - acc: 0.7310 - loss: 0.5507 - val_acc: 0.7650 - val_loss: 0.6096
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 112s 1s/step - acc: 0.8430 - loss: 0.3466 - val_acc: 0.8700 - val_loss: 0.3137
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - acc: 0.8805 - loss: 0.2953 - val_acc: 0.9310 - val_loss: 0.1808
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - acc: 0.8985 - loss: 0.2328 - val_acc: 0.8620 - val_loss: 0.4480
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - acc: 0.9050 - loss: 0.2306 - val_acc: 0.9190 - val_loss: 0.2228
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - acc: 0.9095 - loss: 0.2077 - val_acc: 0.9260 - val_loss: 0.1911
Epoch 7/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 307s 3s/step - acc: 0.9290 - loss: 0.1859 - val_acc: 0.9210 - val_loss: 0.1860
Epoch 8/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - acc: 0.9265 - loss: 0.1850 - val_acc: 0.9380 - val_loss: 0.1561
Epoch 9/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/

In [10]:
test_loss, test_acc = modelIFE.evaluate(test_generator)
print('test acc:', test_acc)

50/50 ━━━━━━━━━━━━━━━━━━━━ 26s 520ms/step - acc: 0.9220 - loss: 0.3029
test acc: 0.921999990940094
